# FUNSD - Exploratory Data Analysis

**Project:** 18 - FUNSD Form Understanding  
**Author:** Sandeep Grover, Liora MLE Cohort 6973  
**Notebook:** `notebooks/01_EDA.ipynb` (NOT executed in Phase 1 scaffold)

This notebook profiles the FUNSD form-understanding dataset before any modelling. We answer six questions:

1. How many forms, entities, and words are in train vs test?
2. What is the class distribution across `question / answer / header / other`?
3. How long are entities (in words and characters)?
4. How are bounding boxes distributed across the page?
5. How dense is the question-to-answer linking graph?
6. What does a single annotated form look like rendered on its image?

## 1. Imports and paths

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_ROOT = Path('../data/dataset')
TRAIN_ANN = DATA_ROOT / 'training_data' / 'annotations'
TRAIN_IMG = DATA_ROOT / 'training_data' / 'images'
TEST_ANN  = DATA_ROOT / 'testing_data'  / 'annotations'
TEST_IMG  = DATA_ROOT / 'testing_data'  / 'images'

print('Train annotations:', len(list(TRAIN_ANN.glob('*.json'))))
print('Test annotations :', len(list(TEST_ANN.glob('*.json'))))

## 2. Load all annotations into a flat DataFrame

One row per entity, with `form_id`, `split`, `entity_id`, `label`, bounding box, text, and word count.

In [ ]:
def load_split(ann_dir, split_name):
    rows = []
    for path in sorted(ann_dir.glob('*.json')):
        form_id = path.stem
        with open(path) as f:
            data = json.load(f)
        for item in data['form']:
            rows.append({
                'split': split_name,
                'form_id': form_id,
                'entity_id': item['id'],
                'label': item['label'],
                'text': item['text'],
                'n_words': len(item['words']),
                'n_chars': len(item['text']),
                'x0': item['box'][0], 'y0': item['box'][1],
                'x1': item['box'][2], 'y1': item['box'][3],
                'n_links': len(item['linking']),
            })
    return pd.DataFrame(rows)

df_train = load_split(TRAIN_ANN, 'train')
df_test  = load_split(TEST_ANN,  'test')
df = pd.concat([df_train, df_test], ignore_index=True)
df.head()

In [ ]:
df.groupby('split').agg(
    n_forms=('form_id', 'nunique'),
    n_entities=('entity_id', 'count'),
    n_words=('n_words', 'sum'),
)

## 3. Class distribution

FUNSD is class-imbalanced: most entities are labelled `other`. Watch for the macro-F1 vs micro-F1 gap when reporting results.

In [ ]:
label_counts = df.groupby(['split', 'label']).size().unstack(fill_value=0)
label_counts['total'] = label_counts.sum(axis=1)
label_counts

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
label_counts.drop(columns='total').T.plot(kind='bar', ax=ax)
ax.set_title('FUNSD class distribution (train vs test)')
ax.set_xlabel('Label')
ax.set_ylabel('Entities')
plt.tight_layout()
plt.show()

## 4. Entity length distribution

Most `header` and `question` entities are short (1-5 words). `answer` entities can be much longer (free-text fills). This drives the choice of max-sequence length in the model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes, ['n_words', 'n_chars'], ['Words per entity', 'Characters per entity']):
    for label in ['question', 'answer', 'header', 'other']:
        ax.hist(df[df.label == label][col], bins=30, alpha=0.5, label=label)
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout()
plt.show()

df.groupby('label')[['n_words', 'n_chars']].describe().round(1)

## 5. Bounding-box geometry

Are headers near the top of the page? Are answers usually wider than questions? Layout signal is exactly what LayoutLMv3 will exploit and what the BERT baseline will ignore.

In [ ]:
df['cx'] = (df.x0 + df.x1) / 2
df['cy'] = (df.y0 + df.y1) / 2
df['width']  = df.x1 - df.x0
df['height'] = df.y1 - df.y0

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, label in zip(axes.flat, ['header', 'question', 'answer', 'other']):
    sub = df[df.label == label]
    ax.scatter(sub.cx, sub.cy, s=4, alpha=0.3)
    ax.invert_yaxis()  # image coords: y increases downward
    ax.set_title(f'{label} (n={len(sub)})')
    ax.set_xlabel('cx (px)')
    ax.set_ylabel('cy (px)')
plt.tight_layout()
plt.show()

## 6. Linking-graph density

How many question-answer pairs does an average form contain? How many answers are unlinked (orphan)? This is the input distribution for the linking head in `model_advanced.py`.

In [ ]:
def build_linking_stats(ann_dir, split):
    rows = []
    for path in sorted(ann_dir.glob('*.json')):
        with open(path) as f:
            data = json.load(f)
        n_q = sum(1 for it in data['form'] if it['label'] == 'question')
        n_a = sum(1 for it in data['form'] if it['label'] == 'answer')
        n_links = sum(len(it['linking']) for it in data['form'])
        rows.append({'form_id': path.stem, 'split': split, 'n_q': n_q, 'n_a': n_a, 'n_links': n_links})
    return pd.DataFrame(rows)

df_links = pd.concat([
    build_linking_stats(TRAIN_ANN, 'train'),
    build_linking_stats(TEST_ANN,  'test'),
])
df_links.groupby('split').describe().round(1)

## 7. Render a single annotated form

Sanity check: overlay entity boxes on the original PNG, colour-coded by label. If this looks reasonable, the loaders are correct.

In [ ]:
from matplotlib.patches import Rectangle

COLOURS = {'question': 'tab:blue', 'answer': 'tab:green', 'header': 'tab:red', 'other': 'tab:gray'}

def render_form(form_id, ann_dir, img_dir):
    with open(ann_dir / f'{form_id}.json') as f:
        data = json.load(f)
    img = Image.open(img_dir / f'{form_id}.png')
    fig, ax = plt.subplots(figsize=(9, 11))
    ax.imshow(img, cmap='gray')
    for item in data['form']:
        x0, y0, x1, y1 = item['box']
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                edgecolor=COLOURS[item['label']], linewidth=1.2))
    ax.set_title(f'Form {form_id}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

sample = sorted(TRAIN_ANN.glob('*.json'))[0].stem
render_form(sample, TRAIN_ANN, TRAIN_IMG)

## 8. Take-aways for modelling

- The class imbalance (`other` dominates) means **macro-F1** is the right headline metric, not accuracy.
- Entity length p95 is small (well under 50 words for `question / header`, larger for `answer`), so a 512-token sequence per form is sufficient for a flattened approach.
- Layout is highly informative: the centroid scatter for `header` clusters near the top of the page, while `question` and `answer` are interleaved in body regions. This is exactly the prior LayoutLMv3 should exploit.
- Linking density is sparse (most question-answer pairs are 1-to-1 within the same row or column). The advanced model should bias the linking-edge classifier toward spatially adjacent pairs.
- Reading order in FUNSD is non-trivial. Token-level evaluation (per the FUNSD paper) sidesteps reading-order issues; entity-level evaluation requires a deterministic tie-breaker.